# Search Evaluation

source: https://www.youtube.com/watch?v=KoyYkv8P_jU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv

In [ ]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index


parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
from evaluation_utils import llm_structured, Questions, llm_structured_retry
from ingest import load_faq_data, build_index

documents = load_faq_data()
documents = [doc for doc in documents if doc['course']== 'llm-zoomcamp']

# note:: we used only llm_zoomcamp data to create ground_truth
df = pd.read_csv('./data/ground_truth-new.csv')
ground_truth = df.to_dict(orient="records")
ground_truth

[{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
  'document': '489dd1c9d9'},
 {'question': 'I saw something about YouTube Live and Slido for questions – can you explain exactly how that works during a live session? Like, what happens when I type a question into Slido?',
  'document': '489dd1c9d9'},
 {'question': 'The announcement channel on Telegram and Slack says there’ll be a video URL posted before the sessions. Will it be recorded afterwards too, so we can watch it later if we miss something?',
  'document': '489dd1c9d9'},
 {'question': "I noticed you mentioned watching live on DataTalksClub's YouTube Channel. Is that the *only* place to see the recordings after they happen, or will there be other options?",
  'document': '489dd1c9d9'},
 {'question': 'Just to clarify – I shouldn’t be asking questions directly in the Zoom chat during the live sessions? What’s the best way to get my question

for each question, we will do search, if search returns one of the document, then the search retrieve the document 

In [19]:
query_example = ground_truth[0]
doc_id =query_example["document"]
query_example

{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
 'document': '489dd1c9d9'}

In [11]:
# use text serach, fit will all doucments, the search
search_index = build_index(documents)

In [17]:
# I want to search this query, and the search gives its document id ('document': '489dd1c9d9'), then it is correct
# because the question is created from answer  '489dd1c9d9'
search_index.search(query_example['question'])

[{'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'},
 {'id': 'd65e05bd7a',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Are there live sessions or office hours for each module?',
  'answer': 'There are no separate live sessions for every module by default. Module materials are pre-recorded and

In [20]:
# create text search function

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return search_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )
results = text_search(query_example['question'])
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')
# now we want to build a matrix, each row has five 

489dd1c9d9 == 489dd1c9d9: True

53f15299b6 == 489dd1c9d9: False

d65e05bd7a == 489dd1c9d9: False

930286278d == 489dd1c9d9: False

610ccb23c0 == 489dd1c9d9: False

In [22]:
matches = list()
for doc in results:
    matches.append(int(doc['id'] == doc_id))
matches

[1, 0, 0, 0, 0]

In [23]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

compute_relevance(query_example, search_function= text_search)

[1, 0, 0, 0, 0]

In [24]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = list()

    for query_with_id in ground_truth:
        relevance = compute_relevance(query_with_id, search_function= search_function)
        relevance_total.append(relevance)

    return relevance_total

relevance_total = compute_relevance_total(ground_truth, text_search)
relevance_total

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0,